# 03 · Random Forest

Se entrena con junio, se evalúa con septiembre y solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

Además del MdAPE sobre toda la captura se registra `mdape_nuevos`, que es el error en los anuncios que no existían en junio (una simulación del caso real, dado que son anuncios nuevos).

El cuaderno tiene dos etapas:

| Etapa | Secciones | Experimento MLflow |
|---|---|---|
| Exploración: conjuntos de variables y un hiperparámetro a la vez | 1 y 2 | `bogoprice-random-forest` |
| Ajuste: combinaciones de los dos hiperparámetros que más aportaron | 3 | `bogoprice-random-forest-ajuste` |

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from bogoprice.experimentos import evaluar, preparar_datos
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Edgar"


# Con False solo se ejecuta la exploracion en las dos variables mas  importantes
EJECUTAR_EXPLORACION = False

datos = preparar_datos()
resultados = []


def rf(**p):
    # Valores base: cada prueba cambia solo los indicados
    base = {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.5, "max_samples": None}
    return RandomForestRegressor(n_jobs=-1, random_state=0, **(base | p)), base | p

## 1. Conjuntos de variables

La configuración base con los tres conjuntos de variables.

In [ ]:
if EJECUTAR_EXPLORACION:
    iniciar_mlflow("bogoprice-random-forest")
    for version in ["v1", "v2", "v2_geo"]:
        modelo, params = rf()
        m = evaluar(datos, f"rf_{version}", modelo, version, {"modelo": "RandomForest", **params}, AUTOR)[0]
        resultados.append({"corrida": f"rf_{version}", "conjunto": version,
                           "parametro": "base", "valor": "-", **m})
        print(f"{version:7s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

## 2. Un hiperparámetro a la vez (v2 con y sin coordenadas)

Se partió de la configuración base (300 árboles, `min_samples_leaf=5`, `max_features=0.5`) y se cambió un hiperparámetro a la vez. Resultados con coordenadas (`v2_geo`, MdAPE en anuncios nuevos; base 18,2 %):

| Hiperparámetro | Valores probados | MdAPE nuevos | ¿Aporta? |
|---|---|---|---|
| `min_samples_leaf` | 1, 10, 20 | 18,10 / 18,26 / **17,98** | Sí |
| `max_samples` | 0,5 | **18,04** | Sí |
| `max_features` | 0,3 / 1,0 | 18,24 / 18,09 | Poco |
| `n_estimators` | 100 / 500 | sin cambio | No |
| `criterion` | poisson | 18,56 | No |
| `bootstrap` | False | 18,07 | No |
| `min_samples_split` | 30 | 18,49 | No (con coordenadas) |
| `max_depth` | 10 / 20 | 19,6 / 18,38 | No, al limitar la profundidad empeora |
| `ccp_alpha` | 0,001 | 22,14 | No dado que subajusta |

Los dos hiperparámetros que más aportaron son **`min_samples_leaf`** y **`max_samples`**: ambos regulan cuántos anuncios terminan en cada hoja y, por tanto, cuánto memoriza el modelo los anuncios de junio. Los demás se dejan en su valor base para la sección 3.

In [ ]:
if EJECUTAR_EXPLORACION:
    pruebas = {
        "n_estimators": [100, 500],
        "max_depth": [10, 20],
        "max_features": [0.3, 1.0],
        "min_samples_leaf": [1, 10, 20],
        "max_samples": [0.5],
        "min_samples_split": [30],
        "bootstrap": [False],
        "criterion": ["poisson"],
        "ccp_alpha": [0.001],
    }
    for version in ["v2", "v2_geo"]:
        for parametro, valores in pruebas.items():
            for valor in valores:
                modelo, params = rf(**{parametro: valor})
                nombre = f"rf_{parametro}{valor}_{version}"
                m = evaluar(datos, nombre, modelo, version,
                            {"modelo": "RandomForest", "parametro_variado": parametro, **params}, AUTOR)[0]
                resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                                   "valor": valor, **m})
    tabla = pd.DataFrame(resultados)
    display(tabla.pivot_table(index=["parametro", "valor"], columns="conjunto",
                              values="mdape_nuevos").round(2))

## 3. Ajuste: `min_samples_leaf` × `max_samples`

Se prueban todas las combinaciones de los dos hiperparámetros que más aportaron, con y sin coordenadas. Los demás quedan en la base (300 árboles, `max_features=0.5`). `max_samples=1.0` significa que cada árbol usa una muestra del mismo tamaño que el entrenamiento (valor por defecto).

In [ ]:
iniciar_mlflow("bogoprice-random-forest-ajuste")

hojas = [5, 10, 20, 30, 50]
muestras = [0.3, 0.5, 0.7, None]

grilla = []
for version in ["v2", "v2_geo"]:
    for hoja in hojas:
        for ms in muestras:
            modelo, params = rf(min_samples_leaf=hoja, max_samples=ms)
            fraccion = 1.0 if ms is None else ms
            nombre = f"rf_leaf{hoja}_samples{fraccion}_{version}"
            m = evaluar(datos, nombre, modelo, version,
                        {"modelo": "RandomForest", "parametro_variado": "grilla", **params}, AUTOR)[0]
            grilla.append({"conjunto": version, "min_samples_leaf": hoja, "max_samples": fraccion, **m})
            print(f"{version:7s} hoja={hoja:<3d} muestras={fraccion:<4} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

grilla = pd.DataFrame(grilla)

In [ ]:
# Mapa de calor del MdAPE en anuncios nuevos para cada conjunto de variables
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
for eje, version in zip(ejes, ["v2", "v2_geo"]):
    tabla = grilla[grilla["conjunto"] == version].pivot(
        index="min_samples_leaf", columns="max_samples", values="mdape_nuevos")
    eje.imshow(tabla.values, cmap="RdYlGn_r", aspect="auto")
    eje.set_xticks(range(len(tabla.columns)), [f"{c:g}" for c in tabla.columns])
    eje.set_yticks(range(len(tabla.index)), tabla.index)
    for i in range(tabla.shape[0]):
        for j in range(tabla.shape[1]):
            eje.text(j, i, f"{tabla.values[i, j]:.2f}", ha="center", va="center", color="black")
    eje.set_xlabel("max_samples (fracción de anuncios por árbol)")
    eje.set_ylabel("min_samples_leaf")
    eje.set_title(f"MdAPE anuncios nuevos (%) — {version}")
fig.tight_layout()

# Se guarda la figura y la tabla en MLflow, en una corrida de resumen
mejores = grilla.loc[grilla.groupby("conjunto")["mdape_nuevos"].idxmin()]
with tempfile.TemporaryDirectory() as tmp:
    ruta_fig = os.path.join(tmp, "grilla_rf_mdape_nuevos.png")
    ruta_csv = os.path.join(tmp, "grilla_rf.csv")
    fig.savefig(ruta_fig, dpi=130)
    grilla.to_csv(ruta_csv, index=False)
    registrar_corrida(
        "rf_resumen_grilla",
        {"min_samples_leaf": str(hojas), "max_samples": str(muestras)},
        {f"mejor_mdape_nuevos_{fila.conjunto}": float(fila.mdape_nuevos) for fila in mejores.itertuples()},
        artefactos=[ruta_fig, ruta_csv],
        tags={"autor": AUTOR, "modelo": "Resumen"},
    )
plt.show()

## Resumen

Las tres mejores combinaciones de cada conjunto de variables.

In [ ]:
columnas = ["conjunto", "min_samples_leaf", "max_samples", "mdape_nuevos", "mdape", "mdape_train"]
grilla.sort_values("mdape_nuevos").groupby("conjunto").head(3)[columnas].round(2)